## Prerequisites 

Runtime: GPU T4. On drive at the specirfied path (to be changed according to the tester's drive) the code expects: hatexplain.jsonl, olid.jsonl, the challenge/ folder with the jsonl files from the tiems, and a preds/ folder to save the results.

The notebook contains 2 experiments: one for the fine-tuned models and one for the LLM

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Executed configurations: 

| # | MODEL | DATA | SEED | TAG |
|---|-------|------|------|-----|
| 1 | bert-base-uncased | hatexplain | 67   | bert |
| 2 | bert-base-uncased | hatexplain | 13   | bert |
| 3 | bert-base-uncased | hatexplain | 1337 | bert |
| 4 | bert-base-uncased | olid       | 67   | bert |
| 5 | GroNLP/hateBERT   | hatexplain | 67   | hatebert |
| 6 | GroNLP/hateBERT   | hatexplain | 13   | hatebert |
| 7 | GroNLP/hateBERT   | hatexplain | 1337 | hatebert |
| 8 | GroNLP/hateBERT   | olid       | 67   | hatebert |

In [ ]:
BASE  = "/content/drive/MyDrive/NLPProject/NLPProject"
MODEL = "GroNLP/hateBERT"
DATA  = "hatexplain"
SEED  = 1337
TAG   = "hatebert"

In [ ]:
import glob
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from sklearn.metrics import f1_score, precision_recall_fscore_support

torch.manual_seed(SEED)
np.random.seed(SEED)

df  = pd.read_json(f"{BASE}/{DATA}.jsonl", lines=True)
tok = AutoTokenizer.from_pretrained(MODEL)

class DS(Dataset):
    def __init__(self, frame):
        self.enc = tok(list(frame["text"]), truncation=True,
                       padding="max_length", max_length=128)
        self.labels = list(frame["label"])
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[i])
        return item

def metrics(pred):
    return {"macro_f1": f1_score(pred.label_ids, pred.predictions.argmax(-1),
                                 average="macro")}

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)

args = TrainingArguments(
    output_dir="/content/models",
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=200,
    seed=SEED,
)

trainer = Trainer(model=model, args=args,
                  train_dataset=DS(df[df.split == "train"]),
                  eval_dataset=DS(df[df.split == "dev"]),
                  compute_metrics=metrics)
trainer.train()

# --- benchmark test set ---
test_df = df[df.split == "test"].copy()
out = trainer.predict(DS(test_df))
test_df["pred"] = out.predictions.argmax(-1)
test_df["prob_hate"] = torch.softmax(torch.tensor(out.predictions), dim=-1)[:, 1].numpy()
test_df.to_json(f"{BASE}/preds/{TAG}_{DATA}_seed{SEED}.jsonl",
                orient="records", lines=True, force_ascii=False)

p, r, f, _ = precision_recall_fscore_support(test_df["label"], test_df["pred"], average=None)
print(f"\n=== {TAG} / {DATA} / seed {SEED} ===")
print(f"macro-F1: {f1_score(test_df['label'], test_df['pred'], average='macro'):.3f}")
print(f"classe 0: P={p[0]:.3f} R={r[0]:.3f} F1={f[0]:.3f}")
print(f"classe 1: P={p[1]:.3f} R={r[1]:.3f} F1={f[1]:.3f}")

# --- challenge set ---
files = sorted(glob.glob(f"{BASE}/challenge/*.jsonl"))
chal = pd.concat([pd.read_json(f, lines=True) for f in files], ignore_index=True)

out_c = trainer.predict(DS(chal))
chal["pred"] = out_c.predictions.argmax(-1)
chal["prob_hate"] = torch.softmax(torch.tensor(out_c.predictions), dim=-1)[:, 1].numpy()
chal.to_json(f"{BASE}/preds/{TAG}_challenge_seed{SEED}.jsonl",
             orient="records", lines=True, force_ascii=False)

neg, pos = chal[chal.label == 0], chal[chal.label == 1]
print(f"\n--- challenge ({len(chal)} item) ---")
print(f"FPR descrittivo: {(neg.pred == 1).mean():.3f}")
print(f"TPR hate:        {(pos.pred == 1).mean():.3f}")
print("\nrecall per explicitness:")
print(pos.groupby("explicitness")["pred"].mean())
print("\nFPR per source_type:")
print(neg.groupby("source_type")["pred"].mean())
print("\nprob_hate media per branch (solo negativi):")
print(neg.groupby("branch")["prob_hate"].agg(["mean", "count"]))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: GroNLP/hateBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1
1,0.362046,0.350181,0.818376
2,0.261783,0.363204,0.835575
3,0.179087,0.437616,0.839301



=== hatebert / hatexplain / seed 1337 ===
macro-F1: 0.827
classe 0: P=0.906 R=0.872 F1=0.889
classe 1: P=0.736 R=0.796 F1=0.765



--- challenge (472 item) ---
FPR descrittivo: 0.007
TPR hate:        0.000

recall per explicitness:
explicitness
coded       0.0
deictic     0.0
explicit    0.0
Name: pred, dtype: float64

FPR per source_type:
source_type
natural     0.000000
template    0.008299
Name: pred, dtype: float64

prob_hate media per branch (solo negativi):
                  mean  count
branch                       
Heer          0.011896     27
Kriegsmarine  0.026228     27
Luftwaffe     0.010260     27
SA            0.008293     27
SS            0.010316     27


In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.9 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

BASE = "/content/drive/MyDrive/NLPProject/NLPProject"
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="auto")
model.eval()

print(next(model.parameters()).device)   # deve stampare cuda:0

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (2 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

cuda:0


In [ ]:
P0 = """You are annotating social media posts for hate speech.

Definition: hate speech is abuse targeted at a protected group, or at an individual because of their membership in that group.

Answer with exactly one word: HATE or NOT.

Post: {text}
Answer:"""

P1 = """You are annotating posts from military collecting forums for hate speech.

Definition: hate speech is abuse targeted at a protected group, or at an individual because of their membership in that group.

These posts describe historical military artifacts. Mentioning insignia, mottos, unit names or ideological symbols is NOT hate speech when the content is attributed to the object, its period, or third parties. It IS hate speech when the author endorses the ideology in their own voice, extends it to the present, denies or minimises associated crimes, or uses the vocabulary against people.

Answer with exactly one word: HATE or NOT.

Post: {text}
Answer:"""

P2 = """You are annotating posts from military collecting forums for hate speech.

Definition: hate speech is abuse targeted at a protected group, or at an individual because of their membership in that group.

These posts describe historical military artifacts. Mentioning insignia, mottos, unit names or ideological symbols is NOT hate speech when the content is attributed to the object, its period, or third parties. It IS hate speech when the author endorses the ideology in their own voice, extends it to the present, denies or minimises associated crimes, or uses the vocabulary against people.

Answer with exactly one word: HATE or NOT.

Examples:

Post: The tunic has the original collar tabs, machine-stitched, typical of 1943 production.
Answer: NOT

Post: Publicly displaying this insignia is restricted in several countries.
Answer: NOT

Post: Picked up a nice piece. The men who wore it had standards nobody matches today.
Answer: HATE

Post: Most of what people repeat about those units was decided by the winners after the war.
Answer: HATE

Post: {text}
Answer:"""


def classify(text, template):
    msg = [{"role": "user", "content": template.format(text=text)}]
    prompt = tok.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    enc = tok(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=5, do_sample=False,
                             pad_token_id=tok.eos_token_id)

    ans = tok.decode(out[0][enc["input_ids"].shape[1]:],
                     skip_special_tokens=True).strip().upper()
    if ans.startswith("HATE"): return 1
    if ans.startswith("NOT"):  return 0
    return None

In [ ]:
import glob
import pandas as pd

files = sorted(glob.glob(f"{BASE}/challenge/*.jsonl"))
chal = pd.concat([pd.read_json(f, lines=True) for f in files], ignore_index=True)
print(len(chal), "item")

def run(template, col):
    preds = []
    for i, t in enumerate(chal.text):
        preds.append(classify(t, template))
        if (i + 1) % 50 == 0:
            print(f"{col}: {i+1}/{len(chal)}")
            chal.loc[:len(preds)-1, col] = preds      # salvataggio parziale
            chal.to_json(f"{BASE}/preds/qwen_challenge.jsonl",
                         orient="records", lines=True, force_ascii=False)
    chal[col] = preds

for template, col in [(P0, "pred_p0"), (P1, "pred_p1"), (P2, "pred_p2")]:
    run(template, col)
    chal.to_json(f"{BASE}/preds/qwen_challenge.jsonl",
                 orient="records", lines=True, force_ascii=False)

for col in ["pred_p0", "pred_p1", "pred_p2"]:
    valid = chal[chal[col].notna()]
    neg, pos = valid[valid.label == 0], valid[valid.label == 1]
    print(f"\n=== {col} ===")
    print(f"non parsabili: {chal[col].isna().sum()}")
    print(f"FPR: {(neg[col] == 1).mean():.3f}   TPR: {(pos[col] == 1).mean():.3f}")
    print(pos.groupby('explicitness')[col].mean())

472 item
pred_p0: 50/472
pred_p0: 100/472
pred_p0: 150/472
pred_p0: 200/472
pred_p0: 250/472
pred_p0: 300/472
pred_p0: 350/472
pred_p0: 400/472
pred_p0: 450/472
pred_p1: 50/472
pred_p1: 100/472
pred_p1: 150/472
pred_p1: 200/472
pred_p1: 250/472
pred_p1: 300/472
pred_p1: 350/472
pred_p1: 400/472
pred_p1: 450/472
pred_p2: 50/472
pred_p2: 100/472
pred_p2: 150/472
pred_p2: 200/472
pred_p2: 250/472
pred_p2: 300/472
pred_p2: 350/472
pred_p2: 400/472
pred_p2: 450/472

=== pred_p0 ===
non parsabili: 0
FPR: 0.010   TPR: 0.072
explicitness
coded       0.000000
deictic     0.000000
explicit    0.230769
Name: pred_p0, dtype: float64

=== pred_p1 ===
non parsabili: 0
FPR: 0.010   TPR: 0.386
explicitness
coded       0.000000
deictic     0.245098
explicit    0.750000
Name: pred_p1, dtype: float64

=== pred_p2 ===
non parsabili: 0
FPR: 0.013   TPR: 0.819
explicitness
coded       0.000000
deictic     0.862745
explicit    0.923077
Name: pred_p2, dtype: float64
